# SRCS Option A - ASR Robustness in Noise

**Raj Mehta - 23BTRCL021 - JAIN (Deemed-to-be University)**  
Speech Recognition and Conversational Systems - Academic Year 2026-27

This notebook runs the existing project modules. Upload the project folder from the top-level My Drive so it lands at `MyDrive/SRCS_Case_Study` (do not create a wrapper folder first). It installs the project requirements, downloads LibriSpeech dev-clean, streams a small set of real MUSAN noise/music/speech clips, runs Whisper tiny.en, and saves outputs in this Drive folder. The downloads and transcription can take time. No synthetic noise is substituted.


In [ ]:
# 1. Mount Google Drive and select the uploaded project folder.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/SRCS_Case_Study')
required = ['README.md', 'config.py', 'requirements.txt', 'src/pipeline.py']
missing = [name for name in required if not (PROJECT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(
        f"Project files missing under {PROJECT_DIR}: {missing}. "
        "Upload the complete SRCS_Case_Study folder to MyDrive first."
    )
%cd /content/drive/MyDrive/SRCS_Case_Study
print('Project:', Path.cwd())

In [ ]:
# 2. Install all project dependencies in this Colab runtime.
!python -m pip install -q -r requirements.txt
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Using CPU. For faster inference, choose Runtime > Change runtime type > T4 GPU, then reconnect.')

In [ ]:
# 3. Run lightweight project tests before downloading datasets or transcribing.
!python -m pytest -q

## Dataset setup and licensing

The pipeline automatically downloads LibriSpeech dev-clean into this Drive project. The official OpenSLR MUSAN file is an 11 GB tarball. Instead, this project streams 15 real recordings for each of the `noise`, `music`, and `speech` configurations from the public `corypaik/musan` Hugging Face mirror, pinned to the commit in `config.py`. Hugging Face Datasets supports streaming, so the full dataset is not downloaded; because the mirror uses large Parquet shards, actual network traffic may exceed the 45 selected WAVs. Only short crops matched to the speech utterance durations are cached in Drive. No synthetic audio is substituted.

OpenSLR lists the corpus under CC BY 4.0, and the mirror documents that individual source recordings can carry their own terms. The pipeline saves source license files and records each selected item's source/path in a manifest. Review those files before redistributing audio or derived materials. See [OpenSLR MUSAN SLR17](https://openslr.org/17/), the [Hugging Face dataset card](https://huggingface.co/datasets/corypaik/musan), and the [Hugging Face streaming documentation](https://huggingface.co/docs/datasets/main/en/stream).

## Experiment run

The next cell runs the existing `src.pipeline` module. It downloads LibriSpeech dev-clean if absent, chooses 15 utterances from distinct speakers using the fixed seed, streams/caches matching MUSAN clips, loads Whisper tiny.en once, detects CUDA automatically, and transcribes clean, noisy (0/10/20 dB), and noise-reduced versions. Each prediction is checkpointed to `results/predictions.csv`; rerunning after interruption resumes completed sample/condition pairs. Keep this browser tab open while the cell runs.

In [ ]:
# 4. Run or resume the full experiment. All files are written inside the Drive project.
!python -u -m src.pipeline --run --allow-download

## Analysis and report outputs

After the experiment finishes, create comparative WER/CER charts, latency chart, condition summary tables, shared-scale spectrogram, and a report draft from actual files. The report remains a draft until manual listening-based error analysis and page-count checks are complete.

In [ ]:
# 5. Generate tables, figures, and report from actual outputs.
!python -m py_compile scripts/generate_report.py
!python -u scripts/analyze_results.py
!python -u scripts/generate_report.py --project-root /content/drive/MyDrive/SRCS_Case_Study --pdf

In [ ]:
# 6. Audit persisted outputs on Drive (this does not rerun inference).
import json, pandas as pd
from pathlib import Path
results = Path('results')
predictions = pd.read_csv(results / 'predictions.csv')
metadata = json.loads((results / 'metadata.json').read_text())
print('Unique utterances:', predictions.sample_id.nunique())
print('Unique speakers:', predictions.speaker_id.nunique())
print('Conditions:', predictions.condition.nunique())
print('Device:', metadata['device'])
print('Prediction rows:', len(predictions))
print('Files:', *[str(p) for p in sorted(results.rglob('*')) if p.is_file()], sep='\n - ')
display(pd.read_csv(results / 'condition_summary.csv'))